# 06 — Cancellation Is a Request

**Target:** Apache Spark 4.2.0 project runtime; Python 3.10+.

Run cells in order. These labs use small local workloads and avoid deliberately exhausting cluster resources. Read each *Why* before changing the experiment.

## Question
Does asking a background task to stop mean its resources have already been released?

**Why this matters:** Future cancellation and thread interruption often signal cancellation; they do not universally prove physical termination. This lab is a safe Python analogue, not a claim that Spark uses identical internals.

In [ ]:
import threading, time
def cooperative(stop):
    while not stop.is_set(): time.sleep(0.02)
def non_cooperative():
    # Bounded demonstration: ignores the stop event until the loop finishes.
    end=time.monotonic()+0.8
    while time.monotonic()<end: pass
    return 'finished'
stop=threading.Event(); t=threading.Thread(target=cooperative,args=(stop,)); t.start()
time.sleep(0.05); stop.set(); t.join(timeout=1)
print('cooperative exited:',not t.is_alive())

In [ ]:
t=threading.Thread(target=non_cooperative); t.start()
print('immediately after start alive:',t.is_alive())
t.join(timeout=0.05); print('after short join alive:',t.is_alive())
t.join(); print('after completion alive:',t.is_alive())

**Expected:** cooperative work can exit soon after receiving a signal; non-cooperative work continues until its bounded operation returns.

**Why:** cancellation has at least two events: request and termination. Production cleanup should close resources only when ownership and task lifecycle make that safe. Never run an unbounded busy loop.